# FINANCE 384 Assignment 1 – Part A

## A.3 Richer Model Selection: Random Forest

Random Forest is selected as the richer model for forecasting next-month stock excess returns.

Relative to pooled OLS, Random Forest can capture nonlinear relationships and interactions among stock characteristics without requiring these relationships to be specified manually. This is potentially useful in cross-sectional return prediction because the association between one characteristic and expected return may depend on other firm characteristics or market conditions.

A Random Forest averages predictions across many regression trees. Each tree is fitted using a bootstrap sample, while only a subset of predictors is considered at each split. This reduces dependence on any single tree structure and can improve stability relative to an individual regression tree.

This notebook establishes the Random Forest model using the same revised A.1 base predictor information used by pooled OLS.

**Important:** A.3 does not perform hyperparameter selection. The reference Random Forest below is fitted on the training sample only. Formal hyperparameter tuning begins in A.4 using the validation sample.


### A.3 modelling protocol

- **Richer model:** Random Forest regression
- **Target:** next-month stock excess return \(r^e_{i,t+1}\)
- **Base information:** same revised A.1 predictor information used by pooled OLS
- **Training sample:** January 1990–December 2014
- **Validation sample:** January 2015–December 2018, reserved for A.4
- **Test sample:** January 2019–November 2022, untouched in A.3
- **A.3 objective:** establish and fit the richer-model class on training data only


In [ ]:
# A.3.1 Imports and fixed settings

import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"

TARGET = "target_ret_excess_tp1"
RANDOM_SEED = 384


In [ ]:
# A.3.2 Load supplied data

panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)

panel["date"] = pd.to_datetime(panel["date"])

print("Panel shape:", panel.shape)
print(
    "Date range:",
    panel["date"].min().date(),
    "to",
    panel["date"].max().date(),
)
print("Unique stocks:", panel["permno"].nunique())
print(
    "Duplicate stock-month rows:",
    panel.duplicated(["permno", "date"]).sum(),
)


In [ ]:
# A.3.3 Define the common revised A.1 predictor information

numeric_predictors = [
    "size",
    "bm",
    "mom12_2",
    "vol12",
    "beta60",
    "ivol60",
    "turnover",
    "dollar_volume",
    "amihud_illiq",
    "divyield",
    "gross_profit",
    "roe",
    "asset_growth",
    "leverage",
    "accruals",
    "mkt_12m",
    "mkt_vol_12m",
    "down_market",
]

continuous_predictors = [
    col for col in numeric_predictors
    if col != "down_market"
]

binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]

print("Numeric predictors:", len(numeric_predictors))
print("Continuous predictors:", len(continuous_predictors))
print("Binary predictor:", binary_predictors)
print("Industry predictor:", categorical_predictors)


### Revised A.1 target construction

The target \(r^e_{i,t+1}\) is constructed using a calendar-month stock join. This ensures that the realised excess return attached to forecast date \(t\) comes from the following calendar month rather than simply the next observed row.


In [ ]:
# A.3.4 Construct the next-calendar-month excess-return target

panel = (
    panel
    .sort_values(["permno", "date"])
    .reset_index(drop=True)
)

panel["month"] = panel["date"].dt.to_period("M")

next_month_return = (
    panel[
        ["permno", "month", "ret_excess_t"]
    ]
    .rename(columns={"ret_excess_t": TARGET})
    .assign(month=lambda df: df["month"] - 1)
)

panel = panel.merge(
    next_month_return,
    on=["permno", "month"],
    how="left",
    validate="one_to_one",
)

print(
    "Rows with valid next-month target:",
    panel[TARGET].notna().sum(),
)


### Revised A.1 missing-data treatment

Missing continuous characteristics are treated using contemporaneous cross-sectional information:

1. create a missingness indicator before imputation;
2. fill missing values using the same-month FF49 industry median;
3. use the same-month market median as fallback.

The missingness indicators are retained as model inputs.


In [ ]:
# A.3.5 Create missingness indicators and impute characteristics

missing_characteristics = [
    col
    for col in continuous_predictors
    if panel[col].isna().any()
]

missing_indicator_columns = []

for col in missing_characteristics:
    indicator = f"{col}_was_missing"

    panel[indicator] = (
        panel[col]
        .isna()
        .astype(int)
    )

    missing_indicator_columns.append(indicator)

    industry_month_median = (
        panel
        .groupby(["month", "ff49_code"])[col]
        .transform("median")
    )

    market_month_median = (
        panel
        .groupby("month")[col]
        .transform("median")
    )

    panel[col] = (
        panel[col]
        .fillna(industry_month_median)
        .fillna(market_month_median)
    )

print(
    "Missingness indicators created:",
    len(missing_indicator_columns),
)

print(
    "Remaining missing continuous values:",
    int(
        panel[continuous_predictors]
        .isna()
        .sum()
        .sum()
    ),
)


In [ ]:
# A.3.6 Apply the prescribed chronological split

analysis = panel.loc[
    panel[TARGET].notna()
].copy()

train = analysis.loc[
    (analysis["date"] >= "1990-01-01")
    & (analysis["date"] <= "2014-12-31")
].copy()

validation = analysis.loc[
    (analysis["date"] >= "2015-01-01")
    & (analysis["date"] <= "2018-12-31")
].copy()

test = analysis.loc[
    (analysis["date"] >= "2019-01-01")
    & (analysis["date"] <= "2022-11-30")
].copy()

sample_summary = pd.DataFrame({
    "Sample": ["Training", "Validation", "Test"],
    "Months": [
        train["month"].nunique(),
        validation["month"].nunique(),
        test["month"].nunique(),
    ],
    "Stock-month rows": [
        len(train),
        len(validation),
        len(test),
    ],
})

sample_summary


### Training feature matrix

A.3 fits the richer model on the training sample only. Validation and test observations are retained only to document the prescribed chronological split; they are not transformed, predicted, scored, or used for model selection in this notebook.


In [ ]:
# A.3.7 Build the training feature matrix

raw_feature_columns = (
    continuous_predictors
    + binary_predictors
    + missing_indicator_columns
    + categorical_predictors
)

X_train_raw = train[
    raw_feature_columns
].copy()

y_train = train[
    TARGET
].to_numpy()

print(
    "Raw predictor columns:",
    len(raw_feature_columns),
)

print(
    "Training observations:",
    len(X_train_raw),
)


### Common preprocessing

To remain consistent with revised A.1 and the pooled OLS benchmark:

- continuous predictors are standardised using parameters estimated from the training sample;
- `down_market` and missingness indicators are passed through unchanged;
- FF49 industry membership is one-hot encoded with one reference category omitted;
- preprocessing is fitted on the training sample only.

Random Forest does not mechanically require standardisation, but retaining the common transformed feature framework keeps the comparison with pooled OLS consistent.


In [ ]:
# A.3.8 Fit preprocessing on training data only

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            StandardScaler(),
            continuous_predictors,
        ),
        (
            "binary",
            "passthrough",
            binary_predictors
            + missing_indicator_columns,
        ),
        (
            "industry",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_predictors,
        ),
    ],
    remainder="drop",
)

preprocessor.fit(X_train_raw)

X_train = preprocessor.transform(
    X_train_raw
)

feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(
    "Training matrix:",
    X_train.shape,
)

print(
    "Transformed predictors:",
    len(feature_names),
)


## Random Forest reference fit

The Random Forest below is fitted only to establish the richer-model class selected in A.3.

The reference specification uses parameter dimensions demonstrated in the course Random Forest workflow:

- `n_estimators = 500`
- `max_depth = 3`
- `min_samples_leaf = 4`
- `max_features = 0.5`

These values are **not** treated as the final selected hyperparameters. A.4 formally compares candidate Random Forest specifications using validation MSE and retains the training-fitted winner.


In [ ]:
# A.3.9 Fit the reference Random Forest on training data only

reference_forest = RandomForestRegressor(
    n_estimators=500,
    max_depth=3,
    min_samples_leaf=4,
    max_features=0.5,
    random_state=RANDOM_SEED,
    n_jobs=-1,
)

reference_forest.fit(
    X_train,
    y_train,
)

print(
    "Reference Random Forest fitted successfully."
)

print(
    "Training observations used:",
    len(y_train),
)

print(
    "Number of trees:",
    reference_forest.n_estimators,
)

print(
    "Reference max_depth:",
    reference_forest.max_depth,
)

print(
    "Reference min_samples_leaf:",
    reference_forest.min_samples_leaf,
)

print(
    "Reference max_features:",
    reference_forest.max_features,
)


### A.3 protocol audit

A.3 should end after establishing the selected richer-model class on training data. Validation-based comparison belongs to A.4, while final out-of-sample assessment belongs to A.5.


In [ ]:
# A.3.10 Final protocol audit

print("A.3 RANDOM FOREST AUDIT")
print("-" * 50)

print(
    "Selected richer-model class:",
    "Random Forest",
)

print(
    "Reference fit sample:",
    "Training only",
)

print(
    "Training observations:",
    len(train),
)

print(
    "Validation observations reserved for A.4:",
    len(validation),
)

print(
    "Test observations reserved for final evaluation:",
    len(test),
)

print(
    "\nValidation predictions generated in A.3:",
    False,
)

print(
    "Validation metric calculated in A.3:",
    False,
)

print(
    "Test predictions generated in A.3:",
    False,
)

print(
    "Final hyperparameters selected in A.3:",
    False,
)

print(
    "\nNext step:",
    "A.4 validation-based Random Forest tuning",
)


## A.3 Summary

Random Forest is selected as the richer model because it can capture nonlinear relationships and interactions among the same base predictor information supplied to pooled OLS.

The revised A.1 target construction, missing-data treatment, missingness indicators, training-based standardisation, and FF49 encoding are retained. A reference Random Forest is then fitted on the January 1990–December 2014 training sample only.

No validation predictions, validation metrics, test predictions, or hyperparameter selection are performed in A.3. Formal hyperparameter selection begins in A.4 using the validation sample.
